# Diagnostic — colab_11 CPT came back INERT (held-out drift 0.0057 << 0.05 gate)

colab_11's runtime is gone and the notebook is frozen; this is a standalone read-only diagnostic that works from the **saved artifacts on Drive** only. It does not re-run any part of the pipeline and produces no committed output.

The near-zero drift has three candidate explanations, which need different responses:

1. **Genuine null** — the LoRA update is honest but barely moves the pooled embedding.
2. **Undertrained** — attention-only r=8 over ~0.67 epoch is too weak a perturbation.
3. **Silent mechanical failure** — gradients never reached the adapter, or the merge never folded the adapter into the base, so the extractor embedded the unmodified base model.

Run one stage at a time.

- **Stage 1** — did the adapter weights move at all? (needs the adapter on Drive; seconds, CPU) → *result: yes, all lora_B non-zero.*
- **Stage 2** — does merging fold that update into the base weights? (re-downloads the base; a few min, CPU) → *result: yes, ~8% relative change in the query weights.*
- **Stage 3** — faithful real-cell reproduction: embed a real sample through base vs merged and triangulate against the saved zero-shot baseline, to decide genuine-null vs a merge/wiring failure. Heavier (re-stands-up the Geneformer env); run in a fresh GPU runtime.

In [ ]:
# Colab preinstalls torchao 0.10.0; PEFT's is_torchao_available() hard-raises
# below its 0.16.0 floor even though nothing here uses it -- remove it (same fix as colab_11 setup).
!pip uninstall -y torchao -q

from google.colab import drive
drive.mount('/content/drive')

ADAPTER_DIR = '/content/drive/MyDrive/ad-glia-fm-prep/geneformer/cpt_aggregated_seed0_adapter'

import os
print('adapter dir contents:')
for f in sorted(os.listdir(ADAPTER_DIR)):
    print(' ', f, os.path.getsize(os.path.join(ADAPTER_DIR, f)), 'bytes')

## Stage 1 — did the adapter weights move?

PEFT initializes LoRA's `B` matrix to exact zero, so at step 0 the adapter has literally no effect. If `lora_B` is still ~0 after training, gradients never reached it (frozen `requires_grad`, optimizer not stepping, etc.) — a mechanical failure, and the drift result is meaningless. If `lora_B` norms are clearly non-zero, training moved the adapter and we proceed to Stage 2.

In [ ]:
from safetensors.torch import load_file

sd = load_file(os.path.join(ADAPTER_DIR, 'adapter_model.safetensors'))

b_norms = []
for name, t in sd.items():
    tag = ''
    if 'lora_B' in name:
        b_norms.append(t.norm().item())
        tag = '  <- B (zero at init)'
    print(f'{name:70s} shape={str(tuple(t.shape)):14s} norm={t.norm().item():.6f} mean_abs={t.abs().mean().item():.6e}{tag}')

print()
if b_norms:
    import numpy as np
    b = np.array(b_norms)
    print(f'lora_B tensors: {len(b)} | norm min={b.min():.6f} median={np.median(b):.6f} max={b.max():.6f}')
    print('VERDICT:', 'B moved -> gradients flowed, go to Stage 2' if b.min() > 1e-6 else 'B still ~0 -> gradients never reached adapter (mechanical failure)')
else:
    print('no lora_B tensors found -- check adapter naming')

## Stage 2 — does merging fold the update into the base weights?

Re-download the exact base recorded in `adapter_config.json`, snapshot a couple of attention-projection weights, apply the adapter, `merge_and_unload()`, and diff. This reproduces colab_11 §6a's merge step. If the merged weights differ from the base by a real (if small) relative amount, the merge mechanism is sound and #3 is ruled out — leaving the genuine-null vs undertrained call. If the delta is exactly 0, the merge is a no-op and the extractor was embedding the base model.

In [ ]:
import json, torch
from transformers import BertForMaskedLM
from peft import PeftModel

cfg = json.load(open(os.path.join(ADAPTER_DIR, 'adapter_config.json')))
base_path = cfg['base_model_name_or_path']
print('base_model_name_or_path:', base_path)
print('target_modules:', cfg.get('target_modules'), '| r:', cfg.get('r'), '| alpha:', cfg.get('lora_alpha'))

# The adapter records the base as a local path from colab_11's (now-gone) runtime.
# Re-fetch just the V2-104M weights to that exact path at colab_11's pinned commit.
if not os.path.exists(os.path.join(base_path, 'config.json')):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='ctheodoris/Geneformer',
                      revision='04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5',
                      allow_patterns=['Geneformer-V2-104M/*'],
                      local_dir='/content/Geneformer')
    print('base model fetched ->', base_path)
else:
    print('base model already present ->', base_path)

base = BertForMaskedLM.from_pretrained(base_path)

# snapshot a few targeted weights BEFORE merge
watch = [n for n, _ in base.named_parameters()
         if n.endswith('attention.self.query.weight')][:2]
before = {n: base.state_dict()[n].detach().clone() for n in watch}

peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = peft_model.merge_and_unload()
msd = merged.state_dict()

print()
for n in watch:
    d = (msd[n] - before[n]).norm().item()
    bn = before[n].norm().item()
    print(f'{n}\n   base_norm={bn:.4f}  delta_norm={d:.6f}  rel_delta={d/bn:.6e}')

any_move = any((msd[n] - before[n]).norm().item() > 0 for n in watch)
print()
print('VERDICT:', 'merge folds the adapter in -> #3 ruled out' if any_move else 'merge is a no-op -> extractor embedded the base model')

## Stage 3 — faithful real-cell reproduction (base vs merged embedding)

Stages 1–2 showed the adapter learned and the merge folds a ~8% weight change into the base, so a silent merge failure is unlikely — but that alone does not prove colab_11 §6a embedded the *merged* checkpoint rather than the base, nor whether 0.0057 sits above the run-to-run noise floor. This stage settles both by reproducing §3a→§6a→§7a on a small real sample and embedding it through **both** the base and the re-merged model, then triangulating against the saved colab_09 zero-shot baseline.

**Run this in a FRESH runtime** (Runtime → Restart session): it re-stands-up colab_11's Geneformer environment. Same numpy-restart dance as colab_11 §1a — after the setup cell installs, restart once, re-run setup (fast the second time), then continue downward. A **GPU + high-RAM** runtime is recommended (EmbExtractor + subset loading). Stages 1–2 above do **not** need to be re-run for this.

Readouts:

- **base-vs-merged drift ≈ 0.005** → the merged model genuinely shifts the pooled embedding by that much → §6a used it → INERT is a *real* null (config too weak, not a bug).
- **base-vs-merged drift ≈ 0** → the 8% weight change washes out of the pooled embedding → §7a's 0.0057 was base-vs-base noise → wiring/void.
- **merged-vs-saved-zeroshot** should reproduce §7a's ~0.0056 if §6a used the merged model; **base-vs-saved-zeroshot ≈ 0** confirms a faithful reproduction and a low cross-session noise floor.

In [ ]:
# --- Stage 3 setup — re-stand-up colab_11's Geneformer environment (mirrors colab_11 §1a) ---
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", "https://github.com/pavlemic/ad-glia-fm-prep.git", REPO_PATH], check=True)
!pip install -q -r {REPO_PATH}/requirements_geneformer.txt

GENEFORMER_REPO = "/content/Geneformer"
GF_COMMIT = "04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5"   # colab_11's pinned geneformer_commit
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
    !git -C {GENEFORMER_REPO} checkout {GF_COMMIT}
!cd {GENEFORMER_REPO} && pip install -q .

# torchao 0.10.0 is preinstalled and unused; PEFT's is_torchao_available() hard-raises below 0.16.0.
!pip uninstall -y torchao -q

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (slower embed)")
print("SETUP DONE -- if this cell upgraded numpy, Runtime>Restart session, re-run this cell, then continue.")

### Stage 3 setup — restart note

If the setup cell above upgraded numpy, **Runtime → Restart session now**, then re-run the setup cell (idempotent and fast the second time) and continue downward. This is the same stale-numpy issue colab_11 §1a documents (Colab's kernel loads numpy before the install overwrites it on disk).

In [ ]:
# --- Stage 3.3 — load a real sample with colab_11's exact cell_index, map Ensembl IDs (§3a + §4a) ---
import os, pickle
import numpy as np, pandas as pd, anndata as ad
import scipy.sparse as sp

DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
N_SAMPLE, SAMPLE_SEED = 2000, 0

# colab_11 §3a builds cell_index = arange over concat([micro, astro]): the first n_micro global
# indices are micro rows (in file order), the rest are astro rows. Reproduce that mapping exactly
# via backed reads so a sampled cell_index lines up with the saved zero-shot baseline. To SLICE the
# astro file we need LOCAL row positions (global - n_micro); the stored cell_index stays GLOBAL.
mb = ad.read_h5ad(MICRO_PATH, backed="r")
ab = ad.read_h5ad(ASTRO_PATH, backed="r")
assert list(mb.var_names) == list(ab.var_names), "gene panels differ between subsets"
n_micro, n_astro = mb.n_obs, ab.n_obs
N = n_micro + n_astro
print("n_micro", n_micro, "| n_astro", n_astro, "| total", N)

rng = np.random.default_rng(SAMPLE_SEED)
g = np.sort(rng.choice(N, size=N_SAMPLE, replace=False))       # global cell_index values
g_micro, g_astro = g[g < n_micro], g[g >= n_micro]

parts = []
if len(g_micro):
    m = mb[g_micro].to_memory()                # micro global index == micro local position
    m.obs = m.obs.copy(); m.obs["cell_index"] = g_micro
    parts.append(m)
if len(g_astro):
    a = ab[g_astro - n_micro].to_memory()      # slice by LOCAL astro position
    a.obs = a.obs.copy(); a.obs["cell_index"] = g_astro   # keep GLOBAL cell_index
    parts.append(a)
samp = ad.concat(parts, join="inner")
samp.obs["cell_index"] = samp.obs["cell_index"].astype(int)
print("sample:", samp.shape)

# raw-counts guard (colab_11 §3a)
data = samp.X.data if sp.issparse(samp.X) else np.asarray(samp.X).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f})"
print("raw-counts int-frac:", round(frac_int, 3))

# §4a: map gene symbols -> Ensembl IDs with Geneformer's own dictionaries; APOE must be tokenizable
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE
with open(ENSEMBL_DICTIONARY_FILE, "rb") as f: symbol_to_ensembl = pickle.load(f)
with open(TOKEN_DICTIONARY_FILE, "rb") as f: token_dictionary = pickle.load(f)
samp.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in samp.var_names]
assert symbol_to_ensembl.get("APOE") in token_dictionary, "APOE not tokenizable -- vocab mismatch"
print("APOE tokenizable: OK |", int(samp.var['ensembl_id'].map(lambda e: e in token_dictionary if e else False).sum()), "genes in vocab")

In [ ]:
# --- Stage 3.4 — tokenize the sample (reproduces colab_11 §4b, incl. the sum_ensembl_ids patch) ---
from geneformer import TranscriptomeTokenizer
import geneformer.tokenizer as _gftok

samp.obs["n_counts"] = np.asarray(samp.X.sum(axis=1)).ravel()
assert (samp.obs["n_counts"] > 0).all(), "zero-count cells present"

TOK_IN, TOK_OUT = "/content/diag_tok_in", "/content/diag_tok_out"
os.makedirs(TOK_IN, exist_ok=True)
os.makedirs(TOK_OUT, exist_ok=True)
samp.write_h5ad(os.path.join(TOK_IN, "samp.h5ad"))

# same upstream-bug patch colab_11 §4b applies: reset var index to RangeIndex after sum_ensembl_ids
_orig_sum = _gftok.sum_ensembl_ids
def _sum_patch(*a, **k):
    r = _orig_sum(*a, **k)
    r.var.index = pd.RangeIndex(r.n_vars)
    return r
_gftok.sum_ensembl_ids = _sum_patch

tk = TranscriptomeTokenizer({"cell_index": "cell_index"}, nproc=4)   # defaults = V2 / GC104M, as §4b
tk.tokenize_data(TOK_IN, TOK_OUT, "diag", file_format="h5ad")
TOK_DS = os.path.join(TOK_OUT, "diag.dataset")
print("tokenized ->", TOK_DS)

In [ ]:
# --- Stage 3.5 — rebuild the merged checkpoint (as §6a did) and embed the sample through base + merged ---
import torch
from transformers import BertForMaskedLM
from peft import PeftModel
from geneformer import EmbExtractor

ADAPTER_DIR = "/content/drive/MyDrive/ad-glia-fm-prep/geneformer/cpt_aggregated_seed0_adapter"
MODEL_DIR   = "/content/Geneformer/Geneformer-V2-104M"      # from the LFS clone in setup
assert os.path.exists(MODEL_DIR), f"base model dir missing: {MODEL_DIR}"

# reproduce §6a: merge the adapter into the base, write a standalone checkpoint
base = BertForMaskedLM.from_pretrained(MODEL_DIR)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
MERGED_DIR = "/content/diag_merged"
merged.save_pretrained(MERGED_DIR)
merged.config.to_json_file(os.path.join(MERGED_DIR, "config.json"))
print("rebuilt merged checkpoint ->", MERGED_DIR)

def _embed(model_dir, name):
    ee = EmbExtractor(model_type="Pretrained", num_classes=0, emb_mode="cell",
                      max_ncells=None, emb_layer=-1, emb_label=["cell_index"],
                      forward_batch_size=64, nproc=4)
    out = f"/content/diag_emb_{name}"
    os.makedirs(out, exist_ok=True)
    return ee.extract_embs(model_dir, TOK_DS, out, name)

df_base   = _embed(MODEL_DIR,  "base")     # same weights colab_09 embedded
df_merged = _embed(MERGED_DIR, "merged")   # the post-CPT checkpoint §6a should have used
print("base emb:", df_base.shape, "| merged emb:", df_merged.shape)

In [ ]:
# --- Stage 3.6 — compare (base vs merged) and triangulate against the saved zero-shot baseline ---
import numpy as np, pandas as pd, scanpy as sc

def _to_X(df):
    cols = [c for c in df.columns if c != "cell_index"]
    return df.set_index("cell_index")[cols].astype(np.float32)

Xb = _to_X(df_base)
Xm = _to_X(df_merged)
common = Xb.index.intersection(Xm.index)
Xb_c = Xb.loc[common].to_numpy()
Xm_c = Xm.loc[common].to_numpy()

def _per_cell_cos(A, B):                       # identical to colab_11 §7a
    num = (A * B).sum(1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12
    return num / den

drift_bm = 1.0 - float(np.median(_per_cell_cos(Xb_c, Xm_c)))
print(f"base vs merged drift (sample, n={len(common)}): {drift_bm:.4f}")

# triangulate against colab_09's saved zero-shot baseline (the vectors §7a diffed against)
ZS_PATH = "/content/drive/MyDrive/ad-glia-fm-prep/geneformer/glia_geneformer_zeroshot.h5ad"
zs = sc.read_h5ad(ZS_PATH)
zs_df = pd.DataFrame(np.asarray(zs.X), index=zs.obs["cell_index"].values).astype(np.float32)
zs_s = zs_df.reindex(common)
if zs_s.notna().all(axis=1).all():
    Xz = zs_s.to_numpy()
    drift_bz = 1.0 - float(np.median(_per_cell_cos(Xb_c, Xz)))
    drift_mz = 1.0 - float(np.median(_per_cell_cos(Xm_c, Xz)))
    print(f"base   vs saved-zeroshot drift: {drift_bz:.4f}   (~0 => faithful base reproduction / low cross-session noise floor)")
    print(f"merged vs saved-zeroshot drift: {drift_mz:.4f}   (should ~ 0.0056 from colab_11 §7a if that run used the merged model)")
else:
    print("some sample cell_index absent from saved zero-shot -- skipping triangulation")

print("\nREADING:")
print(f"  base-vs-merged = {drift_bm:.4f}")
print("  ~0.005 -> merged genuinely shifts the pooled embedding; §6a used it; INERT is a REAL null (config too weak, not a bug)")
print("  ~0     -> the 8% weight change washes out of the pooled embedding; §7a's 0.0057 was noise -> wiring/void")